# 02 · Parse the 10-Q

Stage 2 of the indexing pipeline (see `ARCHITECTURE.md`).

Turns the raw filing (`data/raw/acre-20260630.htm`) into a list of **blocks**, in reading order:

- **text** blocks: one paragraph each
- **table** blocks: one table each, with its rows and a caption taken from the lines just above it

Every block is tagged with its **Part** and **Item** (e.g. *Part I · Item 2. MD&A*), so later answers can cite where they came from.

Output: `data/parsed/acre-20260630.json` (used by the chunk step) and `data/parsed/acre-20260630.md` (for you to read).

In [ ]:
import json
import re
import warnings
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup, Tag, XMLParsedAsHTMLWarning

# SEC filings are inline XBRL (XHTML); parsing them as HTML is intended.
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

RAW_PATH = Path("../data/raw/acre-20260630.htm")
PARSED_DIR = Path("../data/parsed")
PARSED_DIR.mkdir(parents=True, exist_ok=True)
JSON_PATH = PARSED_DIR / "acre-20260630.json"
MD_PATH = PARSED_DIR / "acre-20260630.md"

## Helpers

- `clean`: collapses whitespace and tidies numbers the XBRL tags split up, e.g. `( 137,810 )` → `(137,810)`, `$ 1,108` → `$1,108`
- `iter_blocks`: walks the document in order, yielding each paragraph and each table
- `table_rows`: turns an HTML table into rows of cell text, gluing `$` and `)` / `%` cells onto their numbers

In [ ]:
PART_RE = re.compile(r"^PART\s+(I{1,2})\b", re.I)
ITEM_RE = re.compile(r"^Item\s*(\d+[A-Z]?)\s*[.:]\s*(.+)$", re.I)


def clean(text):
    text = re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+([),%])", r"\1", text)
    return re.sub(r"\$\s+(?=[\d(])", "$", text)


def iter_blocks(node):
    """Yield ("text", str) for leaf divs and ("table", Tag) for tables, in document order."""
    for child in node.children:
        if not isinstance(child, Tag):
            continue
        if child.name == "table":
            yield "table", child
        elif child.find(["div", "table"]):
            yield from iter_blocks(child)
        else:
            text = clean(child.get_text(" "))
            if text:
                yield "text", text


def table_rows(table):
    rows = []
    for tr in table.find_all("tr"):
        cells = [clean(td.get_text(" ")) for td in tr.find_all(["td", "th"])]
        merged = []
        for c in cells:
            if not c:
                continue
            if merged and merged[-1] == "$":
                merged[-1] = "$" + c
            elif merged and re.fullmatch(r"\)?%?\)?", c):
                merged[-1] += c
            else:
                merged.append(c)
        if merged:
            rows.append(merged)
    return rows


def rows_to_text(rows):
    return "\n".join(" | ".join(r) for r in rows)

## Parse

Section headings (`PART I…`, `Item 2.…`) aren't kept as blocks; they set the `part` / `item` tag on the blocks that follow. The table of contents and repeated "Table of Contents" page links are skipped.

In [ ]:
def parse_filing(path):
    soup = BeautifulSoup(Path(path).read_bytes(), "lxml")
    for hidden in soup.find_all("ix:header"):
        hidden.decompose()

    part, item = None, "Cover page"
    blocks, recent = [], []
    for kind, value in iter_blocks(soup.body):
        if kind == "text":
            text = value
            if text.lower() == "table of contents" or text.isdigit():
                continue
            if len(text) < 150 and (m := PART_RE.match(text)):
                part, item = f"Part {m.group(1).upper()}", None
                continue
            if len(text) < 150 and (m := ITEM_RE.match(text)):
                item = f"Item {m.group(1).upper()}. {m.group(2)}"
                continue
            blocks.append({"type": "text", "part": part, "item": item, "text": text})
            recent = (recent + [text])[-2:]
        else:
            rows = table_rows(value)
            if not rows or rows[0][0] == "Page":  # empty layout table or the table of contents
                continue
            blocks.append({
                "type": "table", "part": part, "item": item,
                "caption": " / ".join(t[:200] for t in recent),
                "rows": rows, "text": rows_to_text(rows),
            })
    return blocks


blocks = parse_filing(RAW_PATH)

df = pd.DataFrame(blocks)
df["words"] = df["text"].str.split().str.len()
summary = (
    df.fillna({"part": ""})
    .groupby(["part", "item"], sort=False)
    .agg(text_blocks=("type", lambda s: (s == "text").sum()),
         tables=("type", lambda s: (s == "table").sum()),
         words=("words", "sum"))
)
print(f"{len(blocks)} blocks: {(df.type == 'text').sum()} text, {(df.type == 'table').sum()} tables")
summary

## Spot check

Look at one table and one paragraph to make sure they read correctly.

In [ ]:
balance_sheet = next(b for b in blocks if b["type"] == "table" and "BALANCE SHEETS" in b["caption"])
print("Section:", balance_sheet["part"], "·", balance_sheet["item"])
print("Caption:", balance_sheet["caption"])
print()
print("\n".join(balance_sheet["text"].splitlines()[:15]))

print("\n---\n")
mdna = next(b for b in blocks if b["type"] == "text" and b["item"].startswith("Item 2.") and len(b["text"]) > 300)
print("Section:", mdna["part"], "·", mdna["item"])
print(mdna["text"][:600])

## Save

In [ ]:
JSON_PATH.write_text(json.dumps(blocks, ensure_ascii=False, indent=1), encoding="utf-8")

md_lines, current = [], None
for b in blocks:
    section = " · ".join(x for x in (b["part"], b["item"]) if x)
    if section != current:
        md_lines.append(f"\n## {section}\n")
        current = section
    if b["type"] == "text":
        md_lines.append(b["text"] + "\n")
    else:
        md_lines.append(f"**Table:** {b['caption']}\n\n```\n{b['text']}\n```\n")
MD_PATH.write_text("# ACRE 10-Q · Q2 2026 (parsed)\n" + "\n".join(md_lines), encoding="utf-8")

print("Saved:", JSON_PATH, f"({JSON_PATH.stat().st_size / 1_000:.0f} KB)")
print("Saved:", MD_PATH, f"({MD_PATH.stat().st_size / 1_000:.0f} KB)")